In [383]:
library(readr)
library(dplyr)
library(janitor)
library(stringr)
library(tidyr)
library(stringdist)

In [384]:
scraped <- read_csv("data/brfss_export_data/brfss_2014_phr_1_8_11.csv") |> clean_names()
brfss <- read_csv("data/brfss_all_categories.csv")

Rows: 367 Columns: 56
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (55): feature, question, total_yes_percent, total_no_percent, total_curr...
dbl  (1): area

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 391 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): Sheet, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [385]:
# Go through each column name and clean it up (remove the total and percent tags)
clean_columns <- c()

for (col in colnames(scraped)) {  
  if (col == "area" | col == "question") {
    clean_columns <- c(clean_columns, col)
    next
  }
  if (startsWith(col, "total_")) {
    col <- substring(col, 7)
  }
  if (endsWith(col, "_percent")) {
    col <- substring(col, 1, nchar(col) - 8)
  }
  
  clean_columns <- c(clean_columns, col)
}

colnames(scraped) <- clean_columns      

In [386]:
feature <- table(phr1$feature)
feature <- feature[feature > 1]

reaccuring <- phr1[phr1$feature %in% names(feature), ]
length(reaccuring$feature)
# reaccuring



[1] 66

In [387]:
# rotates the none area and question columns into a row called "response"
scraped <- scraped %>%
  pivot_longer(
    cols      = -c(feature, area, question), 
    names_to  = "response",          
    values_to = "percent"           
  ) %>%
  filter(!is.na(percent))


In [388]:
clean <- function(x) {
  x <- gsub("[^A-Za-z0-9]+", "_", x)   # spaces/punct/newlines -> _
  gsub("^_+|_+$", "", x)               # trim leading/trailing _
}

In [389]:
brfss_features <- brfss[c("Sheet")]

cleaned_brfss <- unlist(lapply(brfss_features [[1]], function(x) tolower(clean(x))))

In [390]:
# Drop the manually-identified bad feature/question matches 
bad <- c(
  "CVD ||| Heart Attack",
  "CVD ||| Stroke",
  "CVD ||| Heart Disease",
  "Mammogram ||| Doctor Ever Recommended a Mammogram",
  "Mammogram ||| Ever Had Clinical Breast Exam",
  "Mammogram ||| Time Since Last Mammogram",
  "Mammogram ||| Miles From Work/Home to Closest Mammogram Clinic",
  "Mammogram ||| Miles From Work/Home Traveled for Most Recent Mammogram",
  "Mammogram ||| Most Recent Mammography Was Recommended",
  "Mammogram Past 2 Yrs 40+ ||| Prostate-Specific Antigen Test in the Past 2 Years, Age 40+",
  "Mammogram Past 2 Yrs 40+ ||| Had A Fall in the Past Year, Age 45+",
  "Mammogram Past 2 Yrs 40+ ||| Clinical Breast Exam in the Past Year, Age 40+",
  "DurationClnscpySgmy ||| Time Since Last Clinical Breast Exam",
  "Diabetes ||| Blood Checked for Sugar",
  "Diabetes ||| Feet Checked for Sores",
  "Diabetes ||| Age When Diagnosed With Diabetes",
  "Diabetes ||| Doctor Checked Feet",
  "Diabetes ||| Doctor Seen for Diabetes",
  "Diabetes ||| Ever Attended Diabetes Education",
  "Diabetes ||| Ever Told Diabetes Affected Eyes",
  "Ever Told High BP ||| Prostate-Specific Antigen Test Ever Recommended",
  "DurationHomeBloodStoolTest ||| Time Since Last Pap Smear Test",
  "DurationHomeBloodStoolTest ||| Time Since Last Prostate-Specific Antigen Test",
  "Health Care Provider ||| Asked in Person or On A Form About Amount of Drinking",
  "Routine Checkup ||| Advised To Reduce or Quit Drinking",
  "FOBT Past Yr ||| Blood Stool Test in the Past 2 Years, Age 50+",
  "FOBT Past Yr ||| Blood Stool Test in the Past 3 Years, Age 50-75",
  "Dr Rx Pain Talk ||| Ever Discussed Prostate-Specific Antigen Test Disadvantages",
  "Dr Rx Pain Talk ||| Ever Discussed Prostate-Specific Antigen Test Advantages",
  "Insulin ||| Taking Medicine or Receiving Treatment for Mental Illness",
  "Binge Drinking ||| Asked in Person or On A Form If Drinking Alcohol",
  "Binge Drinking ||| Asked About Binge Drinking",
  "Difficulty Doing Errands Alo ||| Activity on the Internet: Shopping",
  "Poor Physical Health 5+ Days ||| Days Poor Health Interfered With Usual Activities - 5+ Days",
  "Poor Mental Health 5+ Days ||| Days Poor Mental Health Interfered With Activities - 5+ Days",
  "Poor Mental Health 14+ Days ||| Days Poor Mental Health Interfered With Activities - 14+ Days",
  "Heavy Drinking - Females ||| Binge Drinking or Heavy Alcohol Consumption, Females Age 18-44",
  "Sigm Past 5 Yrs Age 50-75 ||| Sigmoidoscopy in the Past 5 Years and Blood Stool Test in the Past 3 Years, Age 50-75",
  "Any Drinking Past Month ||| Drinking and Driving in the Past Month",
  "Frequency of Smoking ||| Frequency of Seatbelt Use",
  "Heavy Drinking ||| Heavy Alcohol Consumption, Females",
  "Heavy Drinking ||| Heavy Alcohol Consumption, Males",
  "Health Care Provider ||| Have At Least One Personal Doctor"
)

before  <- nrow(scraped)
scraped <- scraped[!paste(scraped$feature, scraped$question, sep = " ||| ") %in% bad, , drop = FALSE]
cat("Dropped", before - nrow(scraped), "rows across", length(bad),
    "bad matches;", nrow(scraped), "rows remain.\n")

Dropped 145 rows across 43 bad matches; 265 rows remain.


In [391]:
scraped$variable <- paste(tolower(clean(scraped$feature)), tolower(clean(scraped$response)), sep = "_")

In [392]:
library(stringdist)
canon <- unique(cleaned_brfss)   # brfss canonical feature_response keys

closest <- function(x) {
  if (length(x) == 0)
    return(data.frame(variable = character(), brfss_key = character(), similarity = numeric()))
  d <- stringdistmatrix(x, canon, method = "jw", p = 0.1)
  data.frame(variable   = x,
             brfss_key  = canon[apply(d, 1, which.min)],
             similarity = round(1 - apply(d, 1, min), 3),
             stringsAsFactors = FALSE)
}

# 1) Auto-snap near-identical variables to the brfss spelling 
no_snap <- c("durationclnscpysgmy_10_or_more_years_ago")     # don't fold 10+ into 5+
cand <- closest(setdiff(unique(scraped$variable), canon))    # only the non-exact ones
snap <- cand[cand$similarity >= 0.93 & !(cand$variable %in% no_snap), ]
cat("Auto-snapped", nrow(snap), "variables -> VERIFY these look correct:\n")
print(snap, row.names = FALSE)
key <- setNames(snap$brfss_key, snap$variable)
scraped$variable <- ifelse(scraped$variable %in% names(key), key[scraped$variable], scraped$variable)

# 2) Manual crosswalk for genuinely different wordings (fill from step 3) 
manual <- c(
  "durationclnscpysgmy_within_the_past_2_years_1_year_but_less_than_2_years_ago" = "durationclnscpysgmy_within_the_past_2_years",
  "durationclnscpysgmy_within_the_past_3_years_2_years_but_less_than_3_years_ago" = "durationclnscpysgmy_within_the_past_3_years",
  "durationclnscpysgmy_within_the_past_5_years_3_years_but_less_than_5_years_ago" = "durationclnscpysgmy_within_the_past_5_years",
  "durationclnscpysgmy_within_the_past_year_anytime_less_than_12_months_ago" = "durationclnscpysgmy_within_the_past_year",
  "diabetes_eye_exam_within_the_past_month_anytime_less_than_1_month_ago" = "diabetes_eye_exam_within_the_past_month",
  "diabetes_eye_exam_within_the_past_2_years_1_year_but_less_than_2_years_ago" = "diabetes_eye_exam_within_the_past_2_years",
  "diabetes_eye_exam_within_the_past_year_1_month_but_less_than_12_months_ago" = "diabetes_eye_exam_within_the_past_year",
  "routine_checkup_within_past_year_anytime_less_than_12_months_ago" = "routine_checkup_within_the_past_year",
  "routine_checkup_within_past_2_years_1_year_but_less_than_2_years_ago" = "routine_checkup_within_the_past_2_years",
  "routine_checkup_within_past_5_years_2_years_but_less_than_5_years_ago" = "routine_checkup_within_the_past_5_years",
  "bmi_4_categories_normal" = "bmi_4_categories_recommended_range",
  "durationclnscpysgmy_within_the_past_10_years_5_years_but_less_than_10_years_ago" = "durationclnscpysgmy_5_or_more_years_ago",
  "diabetes_status_no_pre_diabetes_or_borderline_diabetes"   = "diabetes_no_borderline_or_pre_diabetes",
  "diabetes_status_yes_but_female_told_only_during_pregnancy" = "diabetes_yes_but_during_pregnancy",
  "removeteeth_none"                                          = "teeth_removed_no"
)
scraped$variable <- ifelse(scraped$variable %in% names(manual), manual[scraped$variable], scraped$variable)

# 3) Whatever still doesn't match exactly -> paste these into `manual` above ---
left <- setdiff(scraped$variable, canon)
cat("\nStill unmatched:", length(left), "\n")
print(closest(left)[order(-closest(left)$similarity), ], row.names = FALSE)

Auto-snapped 3 variables -> VERIFY these look correct:
                              variable                         brfss_key
   routine_checkup_5_or_more_years_ago   routine_checkup_5_or_more_years
            smoker_status_never_smoker        smoker_status_never_smoked
 diabetes_eye_exam_2_or_more_years_ago diabetes_eye_exam_2_or_more_years
 similarity
      0.977
      0.985
      0.978

Still unmatched: 1 
                                 variable
 durationclnscpysgmy_10_or_more_years_ago
                               brfss_key similarity
 durationclnscpysgmy_5_or_more_years_ago      0.985


In [393]:
diff <- setdiff(scraped$variable, cleaned_brfss)
cat("Verify that there is not variables in scraped that are not in the brfss dataset \n", diff)

# Remove those that are in the difference
scraped <- scraped[!(scraped$variable %in% diff), ]
setdiff(scraped$variable, cleaned_brfss)

Verify that there is not variables in scraped that are not in the brfss dataset 
 durationclnscpysgmy_10_or_more_years_ago

character(0)

In [394]:
scraped <- scraped %>% mutate(percent_num = parse_number(as.character(percent)))

scraped_long <- scraped %>%
  transmute(variable, phr = area, percent = percent_num) %>%
  arrange(variable, phr)

write_csv(scraped_long, "data/created/brfss_scraped_2014.csv")

In [399]:
test <- read.csv("data/merged_with_svi.csv")

filtered <- test %>% select(ends_with("diff"))

summary(filtered)


 A.One.C...No_pct_diff A.One.C...Yes_pct_diff
 Mode:logical          Mode:logical          
 NA's:254              NA's:254              
                                             
                                             
                                             
                                             
                                             
 Advised.to.Change.Eating.Hab...No_pct_diff
 Mode:logical                              
 NA's:254                                  
                                           
                                           
                                           
                                           
                                           
 Advised.to.Change.Eating.Hab...Yes_pct_diff
 Mode:logical                               
 NA's:254                                   
                                            
                                            
                                            
          